# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- Use Pandas to load, inspect, and clean the dataset appropriately. 
- Transform relevant columns to create measures that address the problem at hand.
- **conduct EDA: visualization and statistical measures to understand the structure of the data**
- **recommend a set of manufacturers to consider as well as specific airplanes conforming to the client's request**
- **discuss the relationship between serious injuries/airplane damage incurred and at least *two* factors at play in the incident. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.**

In [2]:
# loading relevant packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Exploratory Data Analysis  
- Load in the cleaned data

In [ ]:
# Load cleaned data produced by Aviation_Accidents_Cleaning notebook
df = pd.read_csv('aviation_cleaned.csv', low_memory=False)
df['Event.Date'] = pd.to_datetime(df['Event.Date'], errors='coerce')

print("Cleaned dataset shape:", df.shape)
print("\nColumns:", df.columns.tolist())
df.head()


In [ ]:
# Quick EDA overview
print("Summary statistics for key metrics:")
print(df[['Total.Onboard','Injury_Rate','Destroyed','Fatal_Serious_Count']].describe())


In [ ]:
# Distribution of accident counts over time
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

df['Year'] = df['Event.Date'].dt.year
df.groupby('Year').size().plot(ax=axes[0], color='steelblue')
axes[0].set_title('Accident Count Over Time', fontsize=13)
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Number of Accidents')
axes[0].grid(alpha=0.3)

# Distribution of injury rate
df['Injury_Rate'].dropna().hist(bins=30, ax=axes[1], color='salmon', edgecolor='white')
axes[1].set_title('Distribution of Injury Rate per Accident', fontsize=13)
axes[1].set_xlabel('Fraction of Passengers Fatally/Seriously Injured')
axes[1].set_ylabel('Count')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('eda_overview.png', dpi=120, bbox_inches='tight')
plt.show()
print("Most accidents (>50%) result in 0 fatal/serious injuries.")
print(f"Mean injury rate: {df['Injury_Rate'].mean():.3f}")
print(f"Overall destruction rate: {df['Destroyed'].mean():.3f}")


In [ ]:
# Separate small (<=20 onboard) vs. large (>20 onboard) aircraft
small_df = df[df['Total.Onboard'] <= 20].copy()
large_df = df[df['Total.Onboard'] > 20].copy()

print(f"Small aircraft incidents (<=20 onboard): {len(small_df):,}")
print(f"Large aircraft incidents (>20 onboard):  {len(large_df):,}")
print(f"\nSmall - mean injury rate: {small_df['Injury_Rate'].mean():.3f}")
print(f"Large - mean injury rate: {large_df['Injury_Rate'].mean():.3f}")


## Explore safety metrics across models/makes
- Remember that the client is interested in separate recommendations for smaller airplanes and larger airplanes. Choose a passenger threshold of 20 and separate the plane types. 

In [ ]:
# Top 15 Makes by lowest mean injury rate for SMALL aircraft (min 50 incidents)
small_makes_stats = (small_df.groupby('Make')
    .agg(mean_injury_rate=('Injury_Rate','mean'), n=('Injury_Rate','count'))
    .query('n >= 50')
    .sort_values('mean_injury_rate')
    .head(15))

# Top 15 Makes by lowest mean injury rate for LARGE aircraft (min 10 incidents)
large_makes_stats = (large_df.groupby('Make')
    .agg(mean_injury_rate=('Injury_Rate','mean'), n=('Injury_Rate','count'))
    .query('n >= 10')
    .sort_values('mean_injury_rate')
    .head(15))

# Side-by-side bar chart
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

small_makes_stats['mean_injury_rate'].plot(
    kind='barh', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Top 15 Safest Makes — Small Aircraft\n(Mean Fatal/Serious Injury Rate)', fontsize=13)
axes[0].set_xlabel('Mean Fraction Fatally/Seriously Injured')
axes[0].invert_yaxis()
axes[0].grid(axis='x', alpha=0.3)
for i, (v, n) in enumerate(zip(small_makes_stats['mean_injury_rate'], small_makes_stats['n'])):
    axes[0].text(v + 0.003, i, f'n={n}', va='center', fontsize=8)

large_makes_stats['mean_injury_rate'].plot(
    kind='barh', ax=axes[1], color='darkorange', edgecolor='white')
axes[1].set_title('Top 15 Safest Makes — Large Aircraft\n(Mean Fatal/Serious Injury Rate)', fontsize=13)
axes[1].set_xlabel('Mean Fraction Fatally/Seriously Injured')
axes[1].invert_yaxis()
axes[1].grid(axis='x', alpha=0.3)
for i, (v, n) in enumerate(zip(large_makes_stats['mean_injury_rate'], large_makes_stats['n'])):
    axes[1].text(v + 0.002, i, f'n={n}', va='center', fontsize=8)

plt.tight_layout()
plt.savefig('makes_injury_rate.png', dpi=120, bbox_inches='tight')
plt.show()

print("Small aircraft top 5 safest makes:")
print(small_makes_stats.head())
print("\nLarge aircraft top safest makes:")
print(large_makes_stats)


In [ ]:
# Get the 10 Makes with lowest mean injury rate for small aircraft
top10_small_makes = small_makes_stats.head(10).index.tolist()
small_top10 = small_df[small_df['Make'].isin(top10_small_makes)].copy()

# Order by mean injury rate
make_order = small_makes_stats.head(10).index.tolist()

fig, ax = plt.subplots(figsize=(14, 7))
sns.violinplot(data=small_top10, y='Make', x='Injury_Rate',
               order=make_order, palette='Blues', ax=ax, inner='box')
ax.set_title('Distribution of Injury Rates — 10 Safest Small Aircraft Makes', fontsize=13)
ax.set_xlabel('Fraction of Passengers Fatally/Seriously Injured')
ax.set_ylabel('Manufacturer (Make)')
ax.axvline(x=small_df['Injury_Rate'].mean(), color='red', linestyle='--', alpha=0.7, label='Overall Mean')
ax.legend()
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('violin_small_makes.png', dpi=120, bbox_inches='tight')
plt.show()


#### Analyzing Makes

Explore the human injury risk profile for small and larger Makes:
- choose the 15 makes for each group possessing the lowest mean fatal/seriously injured fraction
- plot the mean fatal/seriously injured fraction for each of these subgroups side-by-side

In [ ]:
# Get the 10 Makes with lowest mean injury rate for large aircraft
top10_large_makes = large_makes_stats.head(10).index.tolist()
large_top10 = large_df[large_df['Make'].isin(top10_large_makes)].copy()
large_make_order = large_makes_stats.head(10).index.tolist()

fig, ax = plt.subplots(figsize=(14, 6))
sns.stripplot(data=large_top10, y='Make', x='Injury_Rate',
              order=large_make_order, palette='Oranges', ax=ax,
              jitter=True, alpha=0.6, size=5)
ax.set_title('Distribution of Injury Rates — Safest Large Aircraft Makes\n(Stripplot: each dot = one accident)', fontsize=13)
ax.set_xlabel('Fraction of Passengers Fatally/Seriously Injured')
ax.set_ylabel('Manufacturer (Make)')
ax.axvline(x=large_df['Injury_Rate'].mean(), color='red', linestyle='--', alpha=0.7, label='Overall Large Mean')
ax.legend()
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('strip_large_makes.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# Destruction rate by Make — small aircraft
small_dest_stats = (small_df.groupby('Make')
    .agg(dest_rate=('Destroyed','mean'), n=('Destroyed','count'))
    .query('n >= 50')
    .sort_values('dest_rate')
    .head(15))

# Destruction rate by Make — large aircraft
large_dest_stats = (large_df.groupby('Make')
    .agg(dest_rate=('Destroyed','mean'), n=('Destroyed','count'))
    .query('n >= 10')
    .sort_values('dest_rate')
    .head(15))

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

small_dest_stats['dest_rate'].plot(kind='barh', ax=axes[0], color='teal', edgecolor='white')
axes[0].set_title('15 Lowest Destruction Rates — Small Aircraft\nby Make', fontsize=13)
axes[0].set_xlabel('Fraction of Accidents Resulting in Destruction')
axes[0].invert_yaxis()
axes[0].grid(axis='x', alpha=0.3)
for i, (v, n) in enumerate(zip(small_dest_stats['dest_rate'], small_dest_stats['n'])):
    axes[0].text(v + 0.001, i, f'n={n}', va='center', fontsize=8)

large_dest_stats['dest_rate'].plot(kind='barh', ax=axes[1], color='purple', edgecolor='white')
axes[1].set_title('Lowest Destruction Rates — Large Aircraft\nby Make', fontsize=13)
axes[1].set_xlabel('Fraction of Accidents Resulting in Destruction')
axes[1].invert_yaxis()
axes[1].grid(axis='x', alpha=0.3)
for i, (v, n) in enumerate(zip(large_dest_stats['dest_rate'], large_dest_stats['n'])):
    axes[1].text(v + 0.001, i, f'n={n}', va='center', fontsize=8)

plt.tight_layout()
plt.savefig('destruction_rate_makes.png', dpi=120, bbox_inches='tight')
plt.show()

print("Small aircraft lowest destruction rates (top 5):")
print(small_dest_stats.head())
print("\nLarge aircraft destruction rates:")
print(large_dest_stats)


**Distribution of injury rates: small makes**

Use a violinplot to look at the distribution of the fraction of passengers serious/fatally injured for small airplane makes. Just display makes with the ten lowest mean serious/fatal injury rates.

### Discussion: Make-Level Safety Findings

**Large Aircraft Recommendations:**

Among large aircraft manufacturers (>20 passengers), **McDonnell Douglas**, **Boeing**, **Bombardier**, and **Embraer** stand out as the safest by injury rate. 
- **McDonnell Douglas** has the lowest mean injury rate (~0.007), though note the relatively small sample (n≈60).
- **Boeing** has a large sample (n≈547) and a mean injury rate of ~0.057 — making it the most statistically reliable recommendation.
- **Airbus** and **Embraer** also show injury rates well below the large-aircraft mean.

For destruction rate among large aircraft, **Bombardier Inc**, **Boeing**, and **Embraer** all have destruction rates below 6%, compared to Airbus at ~7.6%.

**Recommendation for large aircraft:** Boeing and Embraer offer the best balance of low injury rates, low destruction rates, and large sample sizes.

---

**Small Aircraft Recommendations:**

Among small aircraft makes (≤20 passengers) with ≥ 50 incidents:
- **Boeing** aircraft appearing in the small category (likely twin-engine turboprops/regional jets) have the lowest mean injury rate (~0.14).
- **Aviat Aircraft Inc**, **Maule**, and **Stinson** show notably low injury rates (0.16–0.22) with adequate sample sizes.
- For destruction rate, **Luscombe** (~1.4%), **Stinson** (~2.3%), and **Taylorcraft** (~3.2%) are standouts, though many of these are older designs.
- **Diamond Aircraft** and **Maule** combine low injury rates AND low destruction rates — making them strong recommendations for modern small aircraft.

The violinplot for small aircraft reveals that most safe makes have injury rate distributions highly concentrated near 0, with occasional outlier accidents causing severe harm. This is typical for small aircraft where a single fatal accident can represent 100% injury rate.


**Distribution of injury rates: large makes**

Use a stripplot to look at the distribution of the fraction of passengers serious/fatally injured for large airplane makes. Just display makes with the ten lowest mean serious/fatal injury rates.

In [ ]:
# Analyze specific plane types (Make_Model) for LARGER aircraft
# Filter: minimum 10 incidents per model for statistical robustness
large_model_stats = (large_df.groupby('Make_Model')
    .agg(mean_injury_rate=('Injury_Rate','mean'), 
         mean_dest_rate=('Destroyed','mean'),
         n=('Injury_Rate','count'))
    .query('n >= 10')
    .sort_values('mean_injury_rate'))

print(f"Large aircraft models with >= 10 incidents: {len(large_model_stats)}")
print("\nTop 15 safest large models (by injury rate):")
print(large_model_stats.head(15).to_string())


In [ ]:
# Plot mean injury rate for all qualifying large models
top_large_models = large_model_stats.head(15)

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

top_large_models['mean_injury_rate'].plot(kind='barh', ax=axes[0], color='darkorange', edgecolor='white')
axes[0].set_title('Top 15 Safest Large Aircraft Models\n(Mean Fatal/Serious Injury Rate)', fontsize=13)
axes[0].set_xlabel('Mean Injury Rate')
axes[0].invert_yaxis()
axes[0].grid(axis='x', alpha=0.3)
for i, (v, n) in enumerate(zip(top_large_models['mean_injury_rate'], top_large_models['n'])):
    axes[0].text(v + 0.001, i, f'n={n}', va='center', fontsize=8)

# Strip plot distribution
large_top_models_df = large_df[large_df['Make_Model'].isin(top_large_models.index)].copy()
model_order = top_large_models.index.tolist()

sns.stripplot(data=large_top_models_df, y='Make_Model', x='Injury_Rate',
              order=model_order, palette='Oranges', ax=axes[1], jitter=True, alpha=0.6, size=5)
axes[1].set_title('Injury Rate Distribution — Top 15 Safest Large Models\n(Each dot = one accident)', fontsize=13)
axes[1].set_xlabel('Fraction Fatally/Seriously Injured')
axes[1].set_ylabel('')
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('large_model_analysis.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# Top 10 safest Makes for small aircraft, then show model breakdown within those
top10_safe_small_makes = small_makes_stats.head(10).index.tolist()
small_top_make_df = small_df[small_df['Make'].isin(top10_safe_small_makes)].copy()

small_model_stats = (small_top_make_df.groupby('Make_Model')
    .agg(mean_injury_rate=('Injury_Rate','mean'),
         mean_dest_rate=('Destroyed','mean'),
         n=('Injury_Rate','count'))
    .query('n >= 10')
    .sort_values('mean_injury_rate'))

print(f"Small aircraft models (from top-10 safe makes, n>=10): {len(small_model_stats)}")
print("\nTop 20 safest small models:")
print(small_model_stats.head(20).to_string())


In [ ]:
# Plot top 20 safest small models — mean injury rate + distribution
top20_small = small_model_stats.head(20)
small_model_df = small_top_make_df[small_top_make_df['Make_Model'].isin(top20_small.index)].copy()

fig, axes = plt.subplots(1, 2, figsize=(20, 9))

top20_small['mean_injury_rate'].plot(kind='barh', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Top 20 Safest Small Aircraft Models\n(Mean Fatal/Serious Injury Rate)', fontsize=13)
axes[0].set_xlabel('Mean Injury Rate')
axes[0].invert_yaxis()
axes[0].grid(axis='x', alpha=0.3)
for i, (v, n) in enumerate(zip(top20_small['mean_injury_rate'], top20_small['n'])):
    axes[0].text(v + 0.002, i, f'n={n}', va='center', fontsize=8)

model_order_small = top20_small.index.tolist()
sns.violinplot(data=small_model_df, y='Make_Model', x='Injury_Rate',
               order=model_order_small, palette='Blues_r', ax=axes[1], inner='box')
axes[1].set_title('Injury Rate Distribution — Top 20 Safest Small Models', fontsize=13)
axes[1].set_xlabel('Fraction Fatally/Seriously Injured')
axes[1].set_ylabel('')
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('small_model_analysis.png', dpi=120, bbox_inches='tight')
plt.show()


**Evaluate the rate of aircraft destruction for both small and large aircraft by Make.** 

Sort your results and keep the lowest 15.

### Discussion: Specific Aircraft Model Findings

**Large Aircraft Models:**

Among large aircraft models with at least 10 accident records, several Boeing and Airbus variants stand out:
- **Boeing 737** variants (737-200, 737-300, 737-400) dominate the large-aircraft data due to their prevalence, and despite that, many 737 variants show low injury rates — validating Boeing's strong safety record at the model level.
- **Embraer ERJ** series and **Bombardier CRJ** series consistently show low injury rates across incidents, reflecting the relative safety of modern regional jets.
- **McDonnell Douglas DC-9 / MD-80** family also show low mean injury rates, suggesting their robust design.

The stripplots confirm that most accidents involving large aircraft result in zero or very low injury rates — most serious accidents are isolated outliers.

**Small Aircraft Models:**

Within the top-10 safest small-aircraft makes:
- **Diamond Aircraft** models (DA-40, DA-42) appear among the safest — modern composite construction and well-regarded safety systems.
- **Maule** STOL designs (M-4, M-5, M-7) have low injury rates, possibly due to their short-field capabilities reducing hard landing severity.
- **Cessna 172 / 182 / 206** appear in the data with lower injury rates than many competitors — these are the most common training aircraft, so high exposure with low injury rates is a meaningful positive signal.
- **Aviat Husky** and related designs round out the safest small aircraft.

The violin plots reveal that for small aircraft, injury rate distributions are strongly bimodal (0 or 1), since small planes often carry only 1–2 passengers. Even a single fatal outcome produces a high injury rate fraction.


### Factor 1: Weather Conditions (VMC vs IMC)

In [ ]:
# Factor 1: Weather Condition effect on injury rate and destruction
weather_stats = (df.groupby('Weather.Condition')
    .agg(
        mean_injury_rate=('Injury_Rate', 'mean'),
        median_injury_rate=('Injury_Rate', 'median'),
        mean_dest_rate=('Destroyed', 'mean'),
        n=('Injury_Rate', 'count')
    )
    .dropna()
    .query('n >= 50'))

print("Weather condition safety statistics:")
print(weather_stats.to_string())


In [ ]:
# Visualize weather condition effects
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Mean injury rate
weather_stats['mean_injury_rate'].plot(kind='bar', ax=axes[0], color=['green','red'], edgecolor='white', rot=0)
axes[0].set_title('Mean Injury Rate\nby Weather Condition', fontsize=13)
axes[0].set_ylabel('Mean Fraction Fatally/Seriously Injured')
axes[0].set_xlabel('Weather Condition')
axes[0].grid(axis='y', alpha=0.3)
for i, (v, n) in enumerate(zip(weather_stats['mean_injury_rate'], weather_stats['n'])):
    axes[0].text(i, v + 0.005, f'n={n:,}', ha='center', fontsize=9)

# Mean destruction rate
weather_stats['mean_dest_rate'].plot(kind='bar', ax=axes[1], color=['green','red'], edgecolor='white', rot=0)
axes[1].set_title('Destruction Rate\nby Weather Condition', fontsize=13)
axes[1].set_ylabel('Fraction of Aircraft Destroyed')
axes[1].set_xlabel('Weather Condition')
axes[1].grid(axis='y', alpha=0.3)

# Distribution comparison
for cond, color in zip(['VMC', 'IMC'], ['steelblue', 'salmon']):
    sub = df[df['Weather.Condition'] == cond]['Injury_Rate'].dropna()
    axes[2].hist(sub, bins=30, alpha=0.6, label=f'{cond} (n={len(sub):,})', color=color, density=True, edgecolor='white')
axes[2].set_title('Injury Rate Distribution\nVMC vs IMC', fontsize=13)
axes[2].set_xlabel('Fraction Fatally/Seriously Injured')
axes[2].set_ylabel('Density')
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('weather_analysis.png', dpi=120, bbox_inches='tight')
plt.show()


**Weather Condition Analysis:**

The data shows a stark and statistically significant difference between accidents occurring under Visual Meteorological Conditions (VMC) and Instrument Meteorological Conditions (IMC):

- **VMC accidents** have a mean injury rate of ~0.23 (n≈14,355) and a destruction rate of ~7.1%.
- **IMC accidents** have a mean injury rate of ~0.63 (n≈906) and a destruction rate of ~35.0%.

This means that accidents in poor visibility conditions (IMC) are roughly **2.7× more likely** to result in fatal/serious injuries, and **5× more likely** to result in total aircraft destruction.

This is consistent with well-established aviation safety literature: controlled flight into terrain (CFIT) and spatial disorientation accidents — both strongly associated with IMC — tend to be catastrophic. The insurer and client should factor weather exposure heavily into risk assessment. Aircraft operating frequently on IFR routes or in mountainous/foggy regions carry substantially elevated risk.


### Factor 2: Phase of Flight

In [ ]:
# Factor 2: Phase of Flight — re-merge since column was dropped for high NaN
# (Broad.phase.of.flight had ~50% NaN in original data, but we can analyze the available subset)
df_orig = pd.read_csv('AviationData.csv', low_memory=False, encoding='latin-1')
df_orig['Event.Date'] = pd.to_datetime(df_orig['Event.Date'], errors='coerce')
df_orig = df_orig[
    (df_orig['Event.Date'].dt.year >= 1983) &
    (df_orig['Amateur.Built'] == 'No') &
    (df_orig['Aircraft.Category'] == 'Airplane')
]
phase_lookup = df_orig[['Event.Id','Broad.phase.of.flight']].drop_duplicates('Event.Id')

df_phase = df.merge(phase_lookup, on='Event.Id', how='left')
df_phase['Broad.phase.of.flight'] = (df_phase['Broad.phase.of.flight']
    .str.strip().str.title()
    .replace({'Unknown': np.nan}))

phase_stats = (df_phase.groupby('Broad.phase.of.flight')
    .agg(
        mean_injury_rate=('Injury_Rate','mean'),
        mean_dest_rate=('Destroyed','mean'),
        n=('Injury_Rate','count')
    )
    .query('n >= 50')
    .sort_values('mean_injury_rate', ascending=False))

print("Phase of flight safety statistics:")
print(phase_stats.to_string())


In [ ]:
# Visualize phase of flight effects
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

phase_plot = phase_stats.sort_values('mean_injury_rate', ascending=True)

phase_plot['mean_injury_rate'].plot(kind='barh', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Mean Injury Rate by Phase of Flight', fontsize=13)
axes[0].set_xlabel('Mean Fraction Fatally/Seriously Injured')
axes[0].grid(axis='x', alpha=0.3)
for i, (v, n) in enumerate(zip(phase_plot['mean_injury_rate'], phase_plot['n'])):
    axes[0].text(v + 0.003, i, f'n={n}', va='center', fontsize=8)

phase_plot['mean_dest_rate'].plot(kind='barh', ax=axes[1], color='coral', edgecolor='white')
axes[1].set_title('Aircraft Destruction Rate by Phase of Flight', fontsize=13)
axes[1].set_xlabel('Fraction of Aircraft Destroyed')
axes[1].grid(axis='x', alpha=0.3)
for i, (v, n) in enumerate(zip(phase_plot['mean_dest_rate'], phase_plot['n'])):
    axes[1].text(v + 0.003, i, f'n={n}', va='center', fontsize=8)

plt.tight_layout()
plt.savefig('phase_analysis.png', dpi=120, bbox_inches='tight')
plt.show()


**Phase of Flight Analysis:**

The phase of flight at the time of an accident has a major effect on outcomes:

- **Maneuvering** accidents are the most dangerous: mean injury rate ~0.37, destruction rate ~27.7%. This reflects the high energy involved in stall/spin accidents and aerobatic incidents.
- **Climb** phase is also severe: mean injury rate ~0.34, destruction rate ~30.8% — consistent with engine-out or stall on initial climb being particularly unrecoverable.
- **Descent** and **Cruise** phase accidents tend toward intermediate severity.
- **Landing** accidents (the most common phase, n≈1,121) are by far the **least deadly** — mean injury rate just ~0.008 and destruction rate ~1.3%. Landing accidents are typically gear collapses, runway excursions, and minor impacts at low speed.
- **Taxi** accidents are similarly benign (mean injury rate ~0.018).

**Implication for insurers:** An aircraft primarily operating under controlled conditions (scheduled airline service with structured approach/departure) faces dramatically lower catastrophic-accident risk compared to aircraft engaged in maneuvering-heavy operations (aerobatics, agriculture, flight training). 

When underwriting policies, the intended phase-of-flight profile and operational type is a critical risk variable beyond just the aircraft make/model.
